In [ ]:
import sys, os
import glob
import json
import random

os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch
torch.set_default_dtype(torch.float64)
torch.set_default_device('cuda:0')
torch.set_printoptions(precision=8)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import openmm.app as app

sys.path.insert(0, '..')
from cmm.ffxml import ForceFieldXML
from cmm.units import HARTREE2KCAL
from cmm.develop.data import EdaData, EspData, DipoleData, PolarizabilityData, slice_data
from cmm.develop.optimize import Trainer, Optimizer
from cmm.develop.metrics import plot_correlation, plot_eda_scan
from cmm.develop.report import report_eda_cluster, report_dipos, report_esp, report_eda_scan
from cmm.develop.extract_params import  extract_multipoles

In [ ]:
extract_multipoles(
    'monomers/methanol/monomer.pdb',
    {
        "O": "o.moh",
        "HO": "ho.moh",
        "C": "c.moh",
        "H1": "hc.moh",
        "H2": "hc.moh",
        "H3": "hc.moh"
    },
    'monomers/methanol/02.gdma/poledit.out'
)

## Data

In [ ]:
dipo_datas = [
    DipoleData.from_files('monomers/methanol/monomer.pdb', glob.glob('monomers/methanol/04.qchem/*/qchem.out')),
]
    
esp_datas = [
    EspData.from_files('monomers/methanol/monomer.pdb', 'monomers/methanol/02.gdma/ESPfitpt.txt', 'monomers/methanol/02.gdma/opt.chg'),

]

pol_datas = [
    PolarizabilityData.from_files('monomers/methanol/monomer.pdb', ['monomers/methanol/01.opt/opt.out'])
]


eda_datas = [
    EdaData.from_files('dimers/methanol_water/struct.pdb', glob.glob('dimers/methanol_water/des370k_qm_opt_dimer*/*/eda.out')),
    EdaData.from_files('dimers/methanol_methanol/struct.pdb', glob.glob('dimers/methanol_methanol/des370k_qm_opt_dimer*/*/eda.out')),
    EdaData.from_files('dimers/methanol_water/struct.pdb', glob.glob('dimers/methanol_water/des*md*/*/eda.out')),
    EdaData.from_files('dimers/methanol_methanol/struct.pdb', glob.glob('dimers/methanol_methanol/des*md*/*/eda.out')),
    EdaData.from_files('trimers/methanol_trimer/struct.pdb', glob.glob('trimers/methanol_trimer/*/eda.out'))
]

eda_datas_batched = 10 * eda_datas[:2]
# # eda_datas_batched = [
# #     eda_datas[0][torch.arange(eda_datas[0].num).expand(10, -1).flatten()],
# #     eda_datas[1][torch.arange(eda_datas[1].num).expand(10, -1).flatten()]
# # ]

# from cmm.units import BOHR2ANG

# md_nmer = EdaData.from_files('dimers/methanol_methanol/struct.pdb', glob.glob('dimers/methanol_methanol/des*md_nmer/*/eda.out'))
# coords = md_nmer.coords * BOHR2ANG
# oodist = torch.norm(coords[:, 1] - coords[:, 7], dim=1)
# md_nmer_data = md_nmer[torch.logical_and(oodist > 3, oodist < 4)]
# eda_datas_batched += 10 * [md_nmer_data]

eda_datas[-1] = eda_datas[-1][eda_datas[-1].energies['total'] < 10]
eda_datas[-2] = eda_datas[-2][eda_datas[-2].energies['total'] < 10]
eda_datas[-3] = eda_datas[-3][eda_datas[-3].energies['total'] < 10]


eda_datas_batched += slice_data(eda_datas[-1], 10)
eda_datas_batched += slice_data(eda_datas[-1], 10)
eda_datas_batched += slice_data(eda_datas[-2], 10)
eda_datas_batched += slice_data(eda_datas[-3], 10)
random.shuffle(eda_datas_batched)
len(eda_datas_batched)

In [ ]:
from cmm.develop.optimize import sqrt_weight_func

ff = ForceFieldXML('water_methanol_refit_9.xml', requires_grad=True)
# ff = ForceFieldXML('/pscratch/sd/e/eric6/pycmm-dev/tests/data/water.xml', requires_grad=True)
optimizer = Optimizer(
    ff,
    freeze_water=True,
    opt_params=[
        # 'Z', 'b_elec', 
        # 'q20', 'q21c', 'q21s', 'q22c', 'q22s'
        'q_pauli', 
        'b_pauli',
        'Kdipo_pauli', 'Kquad_pauli',
        # 'j_cf_pauli'
        # 'b_xpol', 
        # 'q_xpol', 
        # 'Kdipo_xpol', 'Kquad_xpol', 
        # 'alpha_xx', 'alpha_yy', 'alpha_zz', 'eta'
        # 'c0', 'dx', 'dy', 'dz', 
        # 'j_cf', 'j_cf_bb', 'j_cf_angle'
        # 'q_ct_acc', 'q_ct_don', 'Kdipo_ct_acc', 'Kdipo_ct_don', 'Kquad_ct_acc', 'Kquad_ct_don', 'b_ct', 'eps_ct'
        # "b_disp", "C6_disp"
        # "eps_ct", "q_ct_don", "Kdipo_ct_don", "Kquad_ct_don", "q_ct_acc", "Kdipo_ct_acc", "Kquad_ct_acc", "b_ct"
    ],
    optim='adam',
    lr=0.005,
    l2=1000.0,
    l2_params=[
        # 'Z', 
        # 'alpha_xx', 'alpha_yy', 'alpha_zz'
        # 'j_cf', 'j_cf_bb', 'j_cf_angle',
        # 'c0', 'dx', 'dy', 'dz'
        # 'Kdipo_pauli', 'Kquad_pauli',
        # 'q_pauli', 
        # 'b_pauli'
        # "b_xpol", 
        # 'Kdipo_xpol', 'Kquad_xpol'
    ],
    additional_positive_constraints=['q_xpol'],
    # freeze_types=['hc.moh','c.moh']
)

trainer = Trainer(
    ff, 
    optimizer, 
    target_weights={'PolarizabilityData': 1e4, 'EdaData':10, 'EspData': 1e6, 'DipoleData': 1e4},
    eda_weights={'perm_elec': 0.0, 'total': 0.0, 'pauli': 10.0, 'disp': 0.0, 'pol': 0.0, 'ct': 0.0},
    weight_func=None
)

for key in optimizer.opt_params:
    print(key, optimizer.opt_params[key][optimizer.masks[key]].numpy(force=True).tolist())

In [ ]:
report_esp(trainer, esp_datas[0])
report_dipos(trainer, dipo_datas[0])
fig1 = report_eda_cluster(trainer, eda_datas[-1])
fig2 = report_eda_cluster(trainer, eda_datas[-2])
fig3 = report_eda_cluster(trainer, eda_datas[-3])

In [ ]:
train_datas = []
# for i in range(len(eda_datas_batched)):
    # train_datas.append(eda_datas_batched[i])
    # train_datas.append(esp_datas[0])
    # train_datas.append(dipo_datas[0])
    # train_datas.append(pol_datas[0])

for data in eda_datas:
    train_datas.append(data)
    train_datas.append(eda_datas[0])
    train_datas.append(eda_datas[1])
    # train_datas.append(eda_datas[2])

_ = trainer.train(train_datas, 200)

In [ ]:
for key in optimizer.opt_params:
    print(key, optimizer.opt_params[key][optimizer.masks[key]].numpy(force=True).tolist())

In [ ]:
report_esp(trainer, esp_datas[0])
report_dipos(trainer, dipo_datas[0])
fig1 = report_eda_cluster(trainer, eda_datas[-1])
fig2 = report_eda_cluster(trainer, eda_datas[-2])
fig3 = report_eda_cluster(trainer, eda_datas[-3])

In [ ]:
report_eda_cluster(trainer, eda_datas[-2][eda_datas[-2].energies['total'] < -4])

In [ ]:
files = [
    ('dimers/methanol_water/struct.pdb', sorted(glob.glob('dimers/methanol_water/des370k_qm_opt_dimer_1/*/eda.out'), key=lambda x: int(os.path.basename(os.path.dirname(x))[2:]))),
    ('dimers/methanol_water/struct.pdb', sorted(glob.glob('dimers/methanol_water/des370k_qm_opt_dimer_2/*/eda.out'), key=lambda x: int(os.path.basename(os.path.dirname(x))[2:]))),
    ('dimers/methanol_methanol/struct.pdb', sorted(glob.glob('dimers/methanol_methanol/des370k_qm_opt_dimer/*/eda.out'), key=lambda x: int(os.path.basename(os.path.dirname(x))[2:]))),
]

scan_datas = [EdaData.from_files(f[0], f[1]) for f in files]

report_eda_scan(
    trainer, 
    scan_datas[2], 
    xdata=[int(os.path.basename(os.path.dirname(f))[2:]) for f in files[2][1]], 
    xlabel='k',
    keys=['pol'],
    xmin=-5, xmax=10
)

In [ ]:
_ = ff.save('water_methanol_refit_6.xml')

In [ ]:
res, ref, _, _ = trainer.evaluate(eda_datas[-2])
index = torch.argsort(torch.abs(res['total'] - ref['total']), descending=True)[:20]
report_eda_cluster(trainer, eda_datas[-2][index])


In [ ]:
import openmm.app as app
from cmm.units import BOHR2NM, BOHR2ANG

top = eda_datas[-2][index].top
coords = eda_datas[-2][index].coords.numpy(force=True) * BOHR2ANG

for i, coord in enumerate(coords):
    app.PDBFile.writeFile(top, coord, f'{i}.pdb')

In [ ]:
list(glob.glob('dimers/methanol_methanol/des*md*/*/eda.out'))[index[2]]

In [ ]:
report_eda_cluster(trainer, EdaData.from_files('dimers/methanol_methanol/struct.pdb', glob.glob('dimers/methanol_methanol/des370k_md_nmer/*/eda.out')))

In [ ]:
report_eda_cluster(trainer, md_nmer_data)